# Telco Churn Scoring — Deep Dive Tecnico
## Clasificacion, calibracion, lift/gains y priorizacion comercial unificada

**Autor:** Juan Prada | **Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/blastchar/telco-customer-churn) · 7,043 clientes · 21 variables

---

## Que cubre este notebook

1. Setup e imports
2. Carga y validacion (desbalanceo 26.5%)
3. EDA
4. Feature engineering + split train/val/test
5. Hyperparameter tuning (solo sobre train)
6. Calibracion de probabilidades + metricas (F1, PR-AUC, Brier, ECE)
7. Threshold por matriz de costes + lift/gains
8. Interpretabilidad (feature importance / SHAP)
9. Scoring de churn + Case 2 (potencial) + Case 3 (anomalias)
10. Score comercial unificado
11. Conclusiones


## 1. Setup e imports


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_data, validate_data
from src.features.engineering import preprocess, build_features, split_data
from src.models.train import (
    train_evaluate_with_calibration,
    tune_threshold_cost,
    build_cost_curve,
    build_churn_scoring,
)
from src.models.tuning import tune_all_models
from src.models.lift import run_lift_analysis
from src.cases import commercial_potential, anomaly_detection, unified_scoring, survival_analysis
from src.visualization.plots import (
    plot_churn_distribution,
    plot_numeric_by_churn,
    plot_categorical_churn_rate,
    plot_roc_curves,
    plot_pr_curves,
    plot_calibration_curve,
    plot_cost_curve,
    plot_feature_importance,
    plot_shap_summary,
    plot_churn_score_distribution,
    plot_lift_gains,
    plot_potential_scoring,
    plot_anomaly_scoring,
    plot_unified_scoring,
    plot_kaplan_meier,
    plot_cox_hazard_ratios,
    plot_survival_risk_distribution,
)

sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 261
DATA_PATH = Path("../data/telco_churn.csv")
REPORTS = Path("../output/reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("Entorno configurado")

Entorno configurado


## 2. Carga y validacion de datos

El dataset esta **desbalanceado de forma moderada**: ~26.5% churn.
Accuracy no sirve (un modelo naive "siempre No Churn" acierta ~73% sin detectar a nadie).
Por eso usamos F1 / PR-AUC / Recall y `class_weight="balanced"`.


In [2]:
df_raw = load_data(DATA_PATH)
validation_report = validate_data(df_raw)
df_raw.head(3)

Dataset: 7043 filas x 21 columnas
Churn rate: 26.54%
Duplicados: 0
Nulos totales: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
print("Variables numericas:", list(df_raw.select_dtypes("number").columns))
print("\nVariables categoricas:", list(df_raw.select_dtypes("object").columns))
print("\nNota: TotalCharges llega como string. Se convierte en loader.py.")
print("11 clientes con tenure=0 tienen TotalCharges vacio, se imputa a 0.")
print(f"\nChurn rate confirmado: {(df_raw['Churn']=='Yes').mean()*100:.2f}%")

Variables numericas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Variables categoricas: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Nota: TotalCharges llega como string. Se convierte en loader.py.
11 clientes con tenure=0 tienen TotalCharges vacio, se imputa a 0.

Churn rate confirmado: 26.54%


## 3. EDA

### 3.1 Distribucion del target


In [4]:
plot_churn_distribution(df_raw, save_path=None)

### 3.2 Variables numericas vs Churn


In [5]:
plot_numeric_by_churn(df_raw, num_cols=["tenure", "MonthlyCharges", "TotalCharges"], save_path=None)

Insights:
- `tenure`: churners con menor antiguedad → clientes nuevos = mayor riesgo
- `MonthlyCharges`: factura alta asociada a mas churn
- `TotalCharges`: correlacionada con tenure (los que se van pronto acumulan menos)


### 3.3 Categoricas de alto impacto


In [6]:
plot_categorical_churn_rate(
    df_raw,
    cat_cols=["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity"],
    target="Churn",
    save_path=None,
)

Insights:
- `Contract` mes a mes: churn >42%; dos anos: <3% (predictor mas fuerte)
- Fibra optica: mas churn que DSL
- Sin TechSupport / OnlineSecurity: churn mas alto


## 4. Feature engineering + split train/val/test

**Por que train/val/test y no solo train/test?**
- `train`: aprender parametros + tuning de hiperparametros
- `val`: calibrar probabilidades y elegir umbrales (sin tocar test)
- `test`: estimar rendimiento final de forma honesta

Features nuevas relevantes:
- indicadores `has_*` de servicios
- `charge_ratio`, `deviation_from_expected`, `avg_monthly_charge`
- `log_total_charges`, `log_monthly_charges`


In [7]:
df_processed = preprocess(df_raw)
df_features = build_features(df_processed)

X_train, X_val, X_test, y_train, y_val, y_test, cid_train, cid_val, cid_test = split_data(
    df_features, val_size=0.2, test_size=0.2
)

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight (XGBoost) = {scale_pos:.2f}")
print(f"Features finales ({X_train.shape[1]}):")
print(list(X_train.columns))

Train: 4225 muestras | Churn: 26.5%
Val:   1409 muestras | Churn: 26.5%
Test:  1409 muestras  | Churn: 26.5%

scale_pos_weight (XGBoost) = 2.77
Features finales (46):
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'has_onlinesecurity', 'has_onlinebackup', 'has_deviceprotection', 'has_techsupport', 'has_streamingtv', 'has_streamingmovies', 'num_additional_services', 'num_protection_services', 'num_streaming_services', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two 

## 5. Hyperparameter tuning (solo sobre train)

`RandomizedSearchCV` con 5-fold estratificado **exclusivamente en train**.
Val y test quedan intocados para evitar data leakage de seleccion de modelo.


In [8]:
tuned_models, tuning_summary = tune_all_models(
    X_train, y_train, scale_pos_weight=scale_pos, n_iter=12, cv=5
)
tuning_summary

  [Tuning] LogisticRegression — 12 combinaciones x 5 folds ...


    best CV F1 = 0.6220 | params: {'model__solver': 'saga', 'model__penalty': 'l2', 'model__C': 0.5}
  [Tuning] RandomForest — 12 combinaciones x 5 folds ...


    best CV F1 = 0.6216 | params: {'n_estimators': 100, 'min_samples_leaf': 10, 'max_features': 'log2', 'max_depth': 12}
  [Tuning] XGBoost — 12 combinaciones x 5 folds ...


    best CV F1 = 0.6163 | params: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.01, 'colsample_bytree': 0.7}


,Model,best_cv_f1,best_params
0,LogisticRegression,0.6220,"{'model__solver': 'saga', 'model__penalty': 'l..."
1,RandomForest,0.6216,"{'n_estimators': 100, 'min_samples_leaf': 10, ..."
2,XGBoost,0.6163,"{'subsample': 0.8, 'n_estimators': 100, 'min_c..."


## 6. Calibracion + evaluacion

Se calibra con Val (sigmoid / Platt scaling) y se reporta en Test:
- **F1** (metrica principal con desbalanceo)
- **ROC-AUC** y **PR-AUC**
- **Brier** y **ECE** (calidad de la probabilidad)


In [9]:
results, trained_detail = train_evaluate_with_calibration(
    tuned_models, X_train, y_train, X_val, y_val, X_test, y_test
)
results.to_csv(REPORTS / "model_comparison.csv", index=False)
results

  Entrenando LogisticRegression (base) ...
  Calibrando LogisticRegression con Val ...


  Entrenando RandomForest (base) ...
  Calibrando RandomForest con Val ...


  Entrenando XGBoost (base) ...
  Calibrando XGBoost con Val ...


,Model,F1-Score,AUC-ROC,PR-AUC,Precision,Recall,BrierScore,ECE,threshold_opt,risk_threshold_medium,risk_threshold_high,calibration_method
0,RandomForest,0.6606,0.8563,0.6800,0.5725,0.7807,0.1308,0.0164,0.27,0.32,0.47,sigmoid
1,XGBoost,0.6593,0.8575,0.6685,0.5623,0.7968,0.1319,0.0247,0.26,0.32,0.47,sigmoid
2,LogisticRegression,0.6433,0.8534,0.6777,0.5335,0.8102,0.1313,0.0258,0.25,0.32,0.47,sigmoid


In [10]:
calibrated_models = {name: d["calibrated"] for name, d in trained_detail.items()}
plot_roc_curves(calibrated_models, X_test, y_test, save_path=None)
plot_pr_curves(calibrated_models, X_test, y_test, save_path=None)

best_name = results.iloc[0]["Model"]
best_calibrated = trained_detail[best_name]["calibrated"]
best_base = trained_detail[best_name]["base"]
print(f"Mejor modelo: {best_name}")
print(results.iloc[0][["F1-Score", "AUC-ROC", "PR-AUC", "Precision", "Recall", "BrierScore", "ECE"]].to_string())

Mejor modelo: RandomForest
F1-Score      0.6606
AUC-ROC       0.8563
PR-AUC          0.68
Precision     0.5725
Recall        0.7807
BrierScore    0.1308
ECE           0.0164


In [11]:
plot_calibration_curve(best_calibrated, X_test, y_test, model_name=best_name, save_path=None)

Figura guardada en: output/figures/calibration_RandomForest.png


## 7. Threshold por costes + Lift / Gains

Con `cost(FN)=5` y `cost(FP)=1` (perder un cliente cuesta 5x mas que una visita fallida),
el umbral optimo cae por debajo de 0.5. Eso es consecuencia directa del desbalanceo + asimetría de costes.

Lift responde a negocio: *"si contacto el top 10%/20% del score, cuantos churners capturo vs aleatorio?"*


In [12]:
proba_val = best_calibrated.predict_proba(X_val)[:, 1]
cost_result = tune_threshold_cost(y_val, proba_val, cost_fn_fp=1.0, cost_fn_false_neg=5.0)
cost_df = build_cost_curve(y_val, proba_val, cost_fn_fp=1.0, cost_fn_false_neg=5.0)
print(cost_result)
plot_cost_curve(cost_df, optimal_threshold=cost_result["threshold"], save_path=None)

{'threshold': 0.21, 'expected_cost': 555.0, 'fp': 305, 'fn': 50, 'precision': 0.5151033386327504, 'recall': 0.8663101604278075, 'f1': 0.646061814556331, 'cost_fp': 1.0, 'cost_fn': 5.0}


In [13]:
proba_test = best_calibrated.predict_proba(X_test)[:, 1]
lift_result = run_lift_analysis(y_test, proba_test, model_name=best_name, n_bins=10, output_dir=REPORTS)
display(lift_result["lift_table"])
plot_lift_gains(lift_result["lift_table"], model_name=best_name, save_path=None)

  Lift table guardada en: ../output/reports/lift_table_RandomForest.csv
  Top decile lift: 2.85x (churn rate=75.7%)
  Top 10%: captura 50.8% de churners (lift acumulado=2.55x)
  Top 20%: captura 69.0% de churners (lift acumulado=2.30x)
  Top 30%: captura 82.1% de churners (lift acumulado=2.05x)


,decile,n_customers,n_churners,churn_rate,base_rate,lift,cumulative_customers,cumulative_recall,cumulative_lift
0,1,140,106,0.7571,0.2654,2.8524,0.0994,0.2834,2.8524
1,2,141,84,0.5957,0.2654,2.2444,0.1994,0.5080,2.5473
2,3,141,68,0.4823,0.2654,1.8169,0.2995,0.6898,2.3033
3,4,141,49,0.3475,0.2654,1.3092,0.3996,0.8209,2.0543
4,5,141,27,0.1915,0.2654,0.7214,0.4996,0.8930,1.7874
5,6,141,13,0.0922,0.2654,0.3473,0.5997,0.9278,1.5471
6,7,141,16,0.1135,0.2654,0.4275,0.6998,0.9706,1.3870
7,8,141,7,0.0496,0.2654,0.1870,0.7999,0.9893,1.2369
8,9,141,2,0.0142,0.2654,0.0534,0.8999,0.9947,1.1053
9,10,141,2,0.0142,0.2654,0.0534,1.0000,1.0000,1.0000


## 8. Interpretabilidad


In [14]:
if hasattr(best_base, "feature_importances_"):
    plot_feature_importance(best_base, X_train.columns.tolist(), top_n=15, model_name=best_name, save_path=None)
    plot_shap_summary(best_base, X_test, model_name=best_name, save_path=None)
else:
    print("El mejor modelo no expone feature_importances_ directamente (p.ej. LogisticRegression en Pipeline).")

Figura guardada en: output/figures/feature_importance_RandomForest.png


Figura guardada en: output/figures/shap_summary_RandomForest.png


## 9. Scoring de churn + Case 2 + Case 3

### Case 1 — Churn scoring (toda la base)
Tiers dinamicos desde Val (`risk_threshold_medium/high`), no cortes fijos 0.3/0.6.


In [15]:
X_all = df_features.drop(columns=["Churn", "customerID"])
customer_ids_all = df_features["customerID"]

churn_scoring = build_churn_scoring(
    best_calibrated,
    X_all,
    customer_ids=customer_ids_all,
    threshold_medium=trained_detail[best_name]["risk_threshold_medium"],
    threshold_high=trained_detail[best_name]["risk_threshold_high"],
)
churn_scoring.to_csv(REPORTS / "churn_scoring.csv", index=False)
print(churn_scoring["risk_tier"].value_counts())
plot_churn_score_distribution(
    churn_scoring,
    threshold_medium=trained_detail[best_name]["risk_threshold_medium"],
    threshold_high=trained_detail[best_name]["risk_threshold_high"],
    save_path=None,
)
display(churn_scoring.head(10))

risk_tier
Low       4635
High      1685
Medium     723
Name: count, dtype: int64


,customer_id,churn_score,risk_tier,recommended_action
0,7439-DKZTW,0.8561,High,Visita urgente - oferta de retencion personali...
1,9300-RENDD,0.8561,High,Visita urgente - oferta de retencion personali...
2,1761-AEZZR,0.8553,High,Visita urgente - oferta de retencion personali...
3,2514-GINMM,0.8553,High,Visita urgente - oferta de retencion personali...
4,5960-WPXQM,0.8553,High,Visita urgente - oferta de retencion personali...
5,8580-AECUZ,0.8549,High,Visita urgente - oferta de retencion personali...
6,8740-CRYFY,0.8547,High,Visita urgente - oferta de retencion personali...
7,0318-QUUOB,0.8515,High,Visita urgente - oferta de retencion personali...
8,0295-PPHDO,0.8513,High,Visita urgente - oferta de retencion personali...
9,6457-USBER,0.8512,High,Visita urgente - oferta de retencion personali...


### Case 2 — Potencial comercial (regresion)

Target: `monthly_potential = P75(MonthlyCharges | Contract, InternetService) - MonthlyCharges`.
Ranking de upsell por incremento esperado de facturacion.


In [16]:
case2 = commercial_potential.run(df_raw, output_dir=REPORTS)
display(case2["results"])
plot_potential_scoring(case2["scoring"], save_path=None)
display(case2["scoring"].head(10))


=== CASE 2: Potencial Comercial (Regresión) ===


  Target: monthly_potential | media=8.46 | mediana=4.75 | max=53.70
  Clientes con potencial > 0: 5256 (74.6%)
  Entrenando Ridge_baseline ...
  Entrenando XGBoost_Regressor ...


  Ridge_baseline: MAE=2.6438 | RMSE=3.2491 | R²=0.8961


  XGBoost_Regressor: MAE=0.3320 | RMSE=0.5826 | R²=0.9967
  Mejor modelo (MAE): XGBoost_Regressor
  Scoring guardado en: ../output/reports/case2_commercial_scoring.csv
  Distribución prioridad upsell:
upsell_priority
Low       3638
Medium    2312
High      1093


,Model,MAE,RMSE,R2
1,XGBoost_Regressor,0.3320,0.5826,0.9967
0,Ridge_baseline,2.6438,3.2491,0.8961


,customer_id,monthly_potential_eur,upsell_priority
0,0621-JFHOL,52.299999,High
1,9560-BBZXK,51.939999,High
2,7340-KEFQE,49.230000,High
3,3298-QEICA,48.669998,High
4,9236-NDUCW,48.509998,High
5,0191-EQUUH,48.490002,High
6,0458-HEUZG,48.080002,High
7,6122-EFVKN,48.070000,High
8,8069-RHUXK,47.900002,High
9,3884-UEBXB,47.869999,High


### Case 3 — Anomalias de facturacion (Isolation Forest)

Unsupervised: no usa Churn como input. Se valida con Precision@K vs base rate
(como chequeo exploratorio, no como objetivo principal).


In [17]:
case3 = anomaly_detection.run(df_raw, output_dir=REPORTS)
plot_anomaly_scoring(case3["scoring"], save_path=None)
display(case3["scoring"].head(10))


=== CASE 3: Detección de Anomalías en Facturación (Isolation Forest) ===
  Features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_ratio', 'deviation_from_expected', 'avg_monthly_charge', 'num_additional_services', 'price_per_service', 'early_high_charge']
  Clientes: 7043


  Modelo entrenado (contamination=0.05)
  Anomalías detectadas: 353 (5.0%)
  Precision@50: 0.240 (vs base rate 0.265)
  Precision@100: 0.220 (vs base rate 0.265)
  Precision@200: 0.260 (vs base rate 0.265)
  Anomaly score medio churners: 0.3298
  Anomaly score medio no-churners: 0.2683
  Scoring guardado en: ../output/reports/case3_anomaly_scoring.csv


,customer_id,anomaly_score,is_anomaly,churn_real
0,1875-QIVME,1.0000,1,1
1,5709-LVOEQ,0.9989,1,0
2,1371-DWPAZ,0.9892,1,0
3,2276-YDAVZ,0.9699,1,1
4,8647-SDTWQ,0.9557,1,0
5,3370-HXOPH,0.9309,1,0
6,3519-ZKXGG,0.9281,1,1
7,1980-KXVPM,0.9144,1,1
8,4075-WKNIU,0.9088,1,0
9,8879-XUAHX,0.9085,1,0


## 10. Score comercial unificado

Combina:
- `churn_score` (peso 0.50)
- `monthly_potential_eur` (peso 0.30)
- `anomaly_score` (peso 0.20)

y asigna un **playbook** (`Retain_HighValue`, `Grow_Upsell`, etc.).


In [18]:
unified = unified_scoring.run(
    churn_scoring=churn_scoring,
    potential_scoring=case2["scoring"],
    anomaly_scoring=case3["scoring"],
    output_dir=REPORTS,
)
plot_unified_scoring(unified["scoring"], save_path=None)
display(unified["scoring"][[
    "customer_id", "commercial_priority_score", "priority_tier",
    "commercial_segment", "churn_score", "monthly_potential_eur", "anomaly_score",
    "recommended_action",
]].head(15))


=== SCORE COMERCIAL UNIFICADO ===
  Pesos: churn=0.50 | potential=0.30 | anomaly=0.20


  Scoring guardado en: ../output/reports/unified_commercial_scoring.csv
  Distribucion priority_tier:
priority_tier
Low       4930
Medium    1408
High       705
  Distribucion commercial_segment:
commercial_segment
Maintain                  4576
Retain_Urgent             1153
Grow_Upsell                549
Retain_HighValue           422
Nurture_Upsell             122
Investigate_Billing        111
Retain_InvestigateBill     110

  Top 10 por prioridad comercial:
customer_id  commercial_priority_score priority_tier commercial_segment  churn_score  monthly_potential_eur  anomaly_score
 7206-GZCDC                     0.7489          High   Retain_HighValue       0.8507              26.000000         0.5147
 3428-XZMAZ                     0.7473          High   Retain_HighValue       0.8492              26.000000         0.5115
 6457-GIRWB                     0.7473          High   Retain_HighValue       0.8492              26.000000         0.5115
 7660-HDPJV                     0.7462   

,customer_id,commercial_priority_score,priority_tier,commercial_segment,churn_score,monthly_potential_eur,anomaly_score,recommended_action
0,7206-GZCDC,0.7489,High,Retain_HighValue,0.8507,26.000000,0.5147,Retencion urgente + oferta personalizada (clie...
1,3428-XZMAZ,0.7473,High,Retain_HighValue,0.8492,26.000000,0.5115,Retencion urgente + oferta personalizada (clie...
2,6457-GIRWB,0.7473,High,Retain_HighValue,0.8492,26.000000,0.5115,Retencion urgente + oferta personalizada (clie...
3,7660-HDPJV,0.7462,High,Retain_HighValue,0.8462,26.000000,0.5147,Retencion urgente + oferta personalizada (clie...
4,9728-FTTVZ,0.7462,High,Retain_HighValue,0.8462,26.000000,0.5147,Retencion urgente + oferta personalizada (clie...
5,7817-BOQPW,0.7455,High,Retain_HighValue,0.7822,20.080000,0.8724,Retencion urgente + oferta personalizada (clie...
6,0488-GSLFR,0.7454,High,Retain_HighValue,0.8496,25.850000,0.5051,Retencion urgente + oferta personalizada (clie...
7,2276-YDAVZ,0.7452,High,Retain_HighValue,0.7463,20.370001,0.9699,Retencion urgente + oferta personalizada (clie...
8,2636-ALXXZ,0.7447,High,Retain_HighValue,0.8506,25.850000,0.4986,Retencion urgente + oferta personalizada (clie...
9,8375-DKEBR,0.7447,High,Retain_HighValue,0.8506,25.850000,0.4986,Retencion urgente + oferta personalizada (clie...


## 10.bis Survival analysis (Kaplan–Meier + Cox PH)

`duration = tenure`, `event = Churn`. Los no-churners estan right-censored.
Responde *cuando* (riesgo a 6/12/24 meses), no solo *si*.


In [19]:
case4 = survival_analysis.run(df_raw, output_dir=REPORTS)
plot_kaplan_meier(case4["km"]["km_global"], case4["km"]["km_by_contract"], save_path=None)
plot_cox_hazard_ratios(case4["hr_table"], save_path=None)
plot_survival_risk_distribution(case4["scoring"], save_path=None)
display(case4["hr_table"].head(10))
display(case4["scoring"].head(10))


=== CASE 4: Survival Analysis (Kaplan-Meier + Cox PH) ===
  duration=tenure | event=Churn | right-censoring en no-churners
  Clientes: 7043 | eventos: 1869 (26.5%)
  Log-rank (Contract): p=0.00e+00
  Mediana de supervivencia por Contract:
      Contract  median_survival_months    n
Month-to-month                    35.0 3875
      One year                     inf 1473
      Two year                     inf 1695



  Hazard ratios (Cox PH) — top factores de riesgo:
            feature  hazard_ratio  hr_ci_low  hr_ci_high        pvalue
  is_month_to_month      9.208438   7.942833   10.675703 2.214605e-190
   electronic_check      1.626609   1.476127    1.792431  9.012318e-23
           is_fiber      1.438106   1.222644    1.691538  1.147704e-05
          paperless      1.166053   1.046103    1.299757  5.541304e-03
log_monthly_charges      0.913852   0.776203    1.075910  2.794528e-01
     has_dependents      0.903232   0.793682    1.027902  1.228802e-01
          is_senior      0.899847   0.807371    1.002914  5.647187e-02
   has_tech_support      0.656786   0.578738    0.745361  7.363918e-11
  Concordance index: 0.8343
  Scoring guardado en: ../output/reports/survival_risk_scoring.csv
  Distribucion survival_risk_tier (riesgo a 12m):
survival_risk_tier
Low       4816
Medium    1908
High       319


,feature,hazard_ratio,hr_ci_low,hr_ci_high,pvalue
0,is_month_to_month,9.208438,7.942833,10.675703,2.214605e-190
1,electronic_check,1.626609,1.476127,1.792431,9.012318e-23
2,is_fiber,1.438106,1.222644,1.691538,1.147704e-05
3,paperless,1.166053,1.046103,1.299757,5.541304e-03
4,log_monthly_charges,0.913852,0.776203,1.075910,2.794528e-01
5,has_dependents,0.903232,0.793682,1.027902,1.228802e-01
6,is_senior,0.899847,0.807371,1.002914,5.647187e-02
7,has_tech_support,0.656786,0.578738,0.745361,7.363918e-11
8,has_online_security,0.563381,0.496371,0.639438,6.626521e-19
9,has_partner,0.548287,0.494368,0.608086,5.366981e-30


,customer_id,tenure,event_observed,churn_prob_within_6m,churn_prob_within_12m,churn_prob_within_24m,survival_risk_tier
0,3750-CKVKH,2,1,0.3831,0.5160,0.6797,High
1,7855-DIWPO,21,0,0.3827,0.5156,0.6793,High
2,9728-FTTVZ,1,1,0.3825,0.5154,0.6790,High
3,7660-HDPJV,1,1,0.3825,0.5154,0.6790,High
4,3716-BDVDB,1,1,0.3826,0.5154,0.6791,High
5,3453-RTHJQ,6,0,0.3826,0.5154,0.6791,High
6,3428-XZMAZ,1,1,0.3825,0.5153,0.6789,High
7,6457-GIRWB,1,1,0.3825,0.5153,0.6789,High
8,7254-IQWOZ,1,1,0.3824,0.5152,0.6788,High
9,4291-SHSBH,7,0,0.3824,0.5152,0.6789,High


## 11. Conclusiones

### Hallazgos tecnicos
- Desbalanceo **26.5%**: accuracy engañosa; F1 + PR-AUC + calibracion son el marco correcto.
- Con `cost(FN)>>cost(FP)` el umbral optimo baja (~0.21 en la ultima ejecucion).
- Lift fuerte: top decile ~**2.8x**; top 20% captura ~**69%** de churners.
- Calibracion (Brier/ECE) habilita interpretar el score como probabilidad de negocio.

### Output operativo
- Case 1: priorizacion de retencion
- Case 2: priorizacion de upsell
- Case 3: alertas de facturacion atipica
- Unificado: una sola cola comercial con playbooks

### Limitaciones
1. Dataset de telecom; revalidar features en otros sectores.
2. Sin dimension temporal (no survival / no validacion temporal).
3. Anomalias detectan rareza de facturacion, no churn directo (Precision@K ~ base rate).
4. Causalidad no garantizada: las acciones deben validarse con A/B.

### Siguiente iteracion sugerida
- Threshold por capacidad real de visitas del equipo
- Optuna / CV anidado mas exhaustivo
- Survival analysis si aparece historico temporal
